In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T


# ============================================================
# AirOps 360 - Week 4 Task 17
# Bronze Open-Meteo JSON -> Silver hourly weather v0.1
#
# SCOPE:
#   April 2026
#   ORD + ATL only
#
# GRAIN:
#   one airport + one local observation hour
#
# NOT IN SCOPE:
#   full 15-airport backfill
#   additional months
#   flight-weather enrichment
#   Gold
# ============================================================


# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

spark.conf.set("spark.sql.session.timeZone", "UTC")

SOURCE_TABLE = "lh_airops_bronze.brz_weather_api_raw"
TARGET_TABLE = "slv_weather_hourly"

EXPECTED_AIRPORTS = ["ORD", "ATL"]

EXPECTED_BRONZE_ROWS = 2
EXPECTED_HOURS_PER_AIRPORT = 720
EXPECTED_TOTAL_ROWS = 1440

START_DATE = "2026-04-01"
END_DATE = "2026-04-30"

SOURCE_NAME = "open_meteo_historical_weather"

SILVER_VERSION = "0.1"
WEATHER_KEY_VERSION = "weather_key_v1"


BRONZE_LINEAGE_COLS = [
    "_bronze_run_id",
    "_bronze_load_id",
    "_bronze_batch_key",
    "_bronze_contract_version",
    "_bronze_source_name",
    "_bronze_ingested_at_utc",
]


print("TASK 17 CONFIGURATION")
print("---------------------")
print("Source table :", SOURCE_TABLE)
print("Target table :", TARGET_TABLE)
print("Airports     :", EXPECTED_AIRPORTS)
print("Period       :", START_DATE, "to", END_DATE)
print("Expected rows:", EXPECTED_TOTAL_ROWS)


# ============================================================
# 1. DEFINE THE OPEN-METEO JSON CONTRACT
# ============================================================

hourly_schema = T.StructType([
    T.StructField(
        "time",
        T.ArrayType(T.StringType()),
        True,
    ),
    T.StructField(
        "temperature_2m",
        T.ArrayType(T.DoubleType()),
        True,
    ),
    T.StructField(
        "relative_humidity_2m",
        T.ArrayType(T.DoubleType()),
        True,
    ),
    T.StructField(
        "precipitation",
        T.ArrayType(T.DoubleType()),
        True,
    ),
    T.StructField(
        "snowfall",
        T.ArrayType(T.DoubleType()),
        True,
    ),
    T.StructField(
        "weather_code",
        T.ArrayType(T.DoubleType()),
        True,
    ),
    T.StructField(
        "cloud_cover",
        T.ArrayType(T.DoubleType()),
        True,
    ),
    T.StructField(
        "wind_speed_10m",
        T.ArrayType(T.DoubleType()),
        True,
    ),
    T.StructField(
        "wind_direction_10m",
        T.ArrayType(T.DoubleType()),
        True,
    ),
])


hourly_units_schema = T.StructType([
    T.StructField("time", T.StringType(), True),
    T.StructField("temperature_2m", T.StringType(), True),
    T.StructField("relative_humidity_2m", T.StringType(), True),
    T.StructField("precipitation", T.StringType(), True),
    T.StructField("snowfall", T.StringType(), True),
    T.StructField("weather_code", T.StringType(), True),
    T.StructField("cloud_cover", T.StringType(), True),
    T.StructField("wind_speed_10m", T.StringType(), True),
    T.StructField("wind_direction_10m", T.StringType(), True),
])


api_schema = T.StructType([
    T.StructField("latitude", T.DoubleType(), True),
    T.StructField("longitude", T.DoubleType(), True),
    T.StructField("generationtime_ms", T.DoubleType(), True),
    T.StructField("utc_offset_seconds", T.IntegerType(), True),
    T.StructField("timezone", T.StringType(), True),
    T.StructField("timezone_abbreviation", T.StringType(), True),
    T.StructField("elevation", T.DoubleType(), True),
    T.StructField("hourly_units", hourly_units_schema, True),
    T.StructField("hourly", hourly_schema, True),
])


# ============================================================
# 2. READ ONLY THE PROVEN ORD / ATL BRONZE PILOT
# ============================================================

bronze = (
    spark.table(SOURCE_TABLE)
    .filter(
        (F.col("airport_code").isin(EXPECTED_AIRPORTS))
        &
        (F.col("request_start_date") == F.lit(START_DATE).cast("date"))
        &
        (F.col("request_end_date") == F.lit(END_DATE).cast("date"))
        &
        (F.col("_bronze_source_name") == SOURCE_NAME)
    )
)


bronze_rows = bronze.count()

print("\nBronze pilot rows:", bronze_rows)

assert bronze_rows == EXPECTED_BRONZE_ROWS, (
    f"STOP: expected {EXPECTED_BRONZE_ROWS} Bronze response rows "
    f"but found {bronze_rows}"
)


bronze_airports = {
    r["airport_code"]
    for r in bronze.select("airport_code").distinct().collect()
}

assert bronze_airports == set(EXPECTED_AIRPORTS), (
    f"STOP: unexpected airport set: {bronze_airports}"
)


distinct_batch_keys = (
    bronze
    .select("_bronze_batch_key")
    .distinct()
    .count()
)

assert distinct_batch_keys == 2, (
    f"STOP: expected 2 deterministic weather batch keys "
    f"but found {distinct_batch_keys}"
)


missing_lineage = [
    c for c in BRONZE_LINEAGE_COLS
    if c not in bronze.columns
]

assert not missing_lineage, (
    f"STOP: missing Bronze lineage columns: {missing_lineage}"
)


lineage_nulls = (
    bronze
    .agg(
        *[
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in BRONZE_LINEAGE_COLS
        ]
    )
    .first()
    .asDict()
)

assert sum(lineage_nulls.values()) == 0, (
    f"STOP: Bronze lineage contains nulls: {lineage_nulls}"
)

print("BRONZE PILOT VALIDATION: PASS")


# ============================================================
# 3. PARSE THE SOURCE-PRESERVING JSON
# ============================================================

parsed = (
    bronze
    .withColumn(
        "payload",
        F.from_json(
            F.col("response_json"),
            api_schema,
        )
    )
)


json_parse_failures = (
    parsed
    .filter(F.col("payload").isNull())
    .count()
)

assert json_parse_failures == 0, (
    f"STOP: JSON parse failures = {json_parse_failures}"
)


parsed = parsed.select(
    "airport_code",
    "request_start_date",
    "request_end_date",
    "response_timezone",
    "response_utc_offset_seconds",

    *BRONZE_LINEAGE_COLS,

    F.col("payload.latitude").alias(
        "source_latitude"
    ),

    F.col("payload.longitude").alias(
        "source_longitude"
    ),

    F.col("payload.elevation").alias(
        "source_elevation_m"
    ),

    F.col(
        "payload.timezone_abbreviation"
    ).alias(
        "response_timezone_abbreviation"
    ),

    # ----------------------------
    # Source arrays
    # ----------------------------

    F.col("payload.hourly.time").alias(
        "time_arr"
    ),

    F.col(
        "payload.hourly.temperature_2m"
    ).alias(
        "temperature_2m_arr"
    ),

    F.col(
        "payload.hourly.relative_humidity_2m"
    ).alias(
        "relative_humidity_2m_arr"
    ),

    F.col(
        "payload.hourly.precipitation"
    ).alias(
        "precipitation_arr"
    ),

    F.col(
        "payload.hourly.snowfall"
    ).alias(
        "snowfall_arr"
    ),

    F.col(
        "payload.hourly.weather_code"
    ).alias(
        "weather_code_arr"
    ),

    F.col(
        "payload.hourly.cloud_cover"
    ).alias(
        "cloud_cover_arr"
    ),

    F.col(
        "payload.hourly.wind_speed_10m"
    ).alias(
        "wind_speed_10m_arr"
    ),

    F.col(
        "payload.hourly.wind_direction_10m"
    ).alias(
        "wind_direction_10m_arr"
    ),

    # ----------------------------
    # Source unit metadata
    # ----------------------------

    F.col(
        "payload.hourly_units.temperature_2m"
    ).alias(
        "unit_temperature_2m"
    ),

    F.col(
        "payload.hourly_units.relative_humidity_2m"
    ).alias(
        "unit_relative_humidity_2m"
    ),

    F.col(
        "payload.hourly_units.precipitation"
    ).alias(
        "unit_precipitation"
    ),

    F.col(
        "payload.hourly_units.snowfall"
    ).alias(
        "unit_snowfall"
    ),

    F.col(
        "payload.hourly_units.weather_code"
    ).alias(
        "unit_weather_code"
    ),

    F.col(
        "payload.hourly_units.cloud_cover"
    ).alias(
        "unit_cloud_cover"
    ),

    F.col(
        "payload.hourly_units.wind_speed_10m"
    ).alias(
        "unit_wind_speed_10m"
    ),

    F.col(
        "payload.hourly_units.wind_direction_10m"
    ).alias(
        "unit_wind_direction_10m"
    ),
)


print("\nJSON PARSING: PASS")


# ============================================================
# 4. VALIDATE RESPONSE TIMEZONE + ARRAY CARDINALITY
# ============================================================

response_structure = (
    parsed
    .select(
        "airport_code",
        "response_timezone",
        "response_utc_offset_seconds",

        F.size("time_arr").alias("time_count"),
        F.size("temperature_2m_arr").alias("temperature_count"),
        F.size("relative_humidity_2m_arr").alias("humidity_count"),
        F.size("precipitation_arr").alias("precipitation_count"),
        F.size("snowfall_arr").alias("snowfall_count"),
        F.size("weather_code_arr").alias("weather_code_count"),
        F.size("cloud_cover_arr").alias("cloud_cover_count"),
        F.size("wind_speed_10m_arr").alias("wind_speed_count"),
        F.size("wind_direction_10m_arr").alias("wind_direction_count"),
    )
)

display(response_structure)


bad_hour_counts = (
    response_structure
    .filter(
        F.col("time_count") != EXPECTED_HOURS_PER_AIRPORT
    )
    .count()
)

assert bad_hour_counts == 0, (
    "STOP: one or more airport responses do not contain "
    f"{EXPECTED_HOURS_PER_AIRPORT} hours"
)


ARRAY_COUNT_COLS = [
    "temperature_count",
    "humidity_count",
    "precipitation_count",
    "snowfall_count",
    "weather_code_count",
    "cloud_cover_count",
    "wind_speed_count",
    "wind_direction_count",
]


for count_col in ARRAY_COUNT_COLS:

    mismatches = (
        response_structure
        .filter(
            F.col(count_col) != F.col("time_count")
        )
        .count()
    )

    assert mismatches == 0, (
        f"STOP: {count_col} does not match time-array cardinality"
    )


timezone_nulls = (
    response_structure
    .filter(
        F.col("response_timezone").isNull()
        |
        F.col("response_utc_offset_seconds").isNull()
    )
    .count()
)

assert timezone_nulls == 0, (
    "STOP: timezone metadata contains NULL values"
)


print("TIMEZONE + ARRAY CARDINALITY VALIDATION: PASS")


# ============================================================
# 5. VALIDATE OPEN-METEO UNITS BEFORE RENAMING VARIABLES
# ============================================================

EXPECTED_UNITS = {
    "unit_temperature_2m": "°C",
    "unit_relative_humidity_2m": "%",
    "unit_precipitation": "mm",
    "unit_snowfall": "cm",
    "unit_weather_code": "wmo code",
    "unit_cloud_cover": "%",
    "unit_wind_speed_10m": "km/h",
    "unit_wind_direction_10m": "°",
}


print("\nOPEN-METEO UNIT VALIDATION")

for unit_col, expected_unit in EXPECTED_UNITS.items():

    actual_values = [
        r[0]
        for r in (
            parsed
            .select(unit_col)
            .distinct()
            .collect()
        )
    ]

    print(
        f"{unit_col}: "
        f"{actual_values}"
    )

    invalid = (
        parsed
        .filter(
            F.col(unit_col).isNull()
            |
            (F.col(unit_col) != expected_unit)
        )
        .count()
    )

    assert invalid == 0, (
        f"STOP: unexpected unit for {unit_col}; "
        f"expected {expected_unit}"
    )


print("UNIT VALIDATION: PASS")


# ============================================================
# 6. EXPLODE EACH AIRPORT/MONTH RESPONSE INTO HOURLY ROWS
# ============================================================

exploded = (
    parsed
    .select(
        "*",
        F.posexplode(
            F.col("time_arr")
        ).alias(
            "source_hour_index",
            "weather_hour_local_text",
        )
    )
)


hourly = (
    exploded

    # ----------------------------
    # Typed local timestamp
    # ----------------------------

    .withColumn(
        "weather_hour_local",
        F.to_timestamp(
            F.col("weather_hour_local_text"),
            "yyyy-MM-dd'T'HH:mm",
        )
    )

    # ----------------------------
    # UTC timestamp using IANA TZ
    #
    # This is preferred over blindly
    # applying a fixed numeric offset
    # because IANA zones handle DST.
    # ----------------------------

    .withColumn(
        "weather_hour_utc",
        F.expr(
            """
            to_utc_timestamp(
                weather_hour_local,
                response_timezone
            )
            """
        )
    )

    # ----------------------------
    # Extract same-index measurements
    # ----------------------------

    .withColumn(
        "temperature_2m_c",
        F.element_at(
            F.col("temperature_2m_arr"),
            F.col("source_hour_index") + F.lit(1),
        ).cast("double")
    )

    .withColumn(
        "relative_humidity_2m_pct",
        F.element_at(
            F.col("relative_humidity_2m_arr"),
            F.col("source_hour_index") + F.lit(1),
        ).cast("double")
    )

    .withColumn(
        "precipitation_mm",
        F.element_at(
            F.col("precipitation_arr"),
            F.col("source_hour_index") + F.lit(1),
        ).cast("double")
    )

    .withColumn(
        "snowfall_cm",
        F.element_at(
            F.col("snowfall_arr"),
            F.col("source_hour_index") + F.lit(1),
        ).cast("double")
    )

    .withColumn(
        "weather_code",
        F.element_at(
            F.col("weather_code_arr"),
            F.col("source_hour_index") + F.lit(1),
        ).cast("int")
    )

    .withColumn(
        "cloud_cover_pct",
        F.element_at(
            F.col("cloud_cover_arr"),
            F.col("source_hour_index") + F.lit(1),
        ).cast("double")
    )

    .withColumn(
        "wind_speed_10m_kmh",
        F.element_at(
            F.col("wind_speed_10m_arr"),
            F.col("source_hour_index") + F.lit(1),
        ).cast("double")
    )

    .withColumn(
        "wind_direction_10m_deg",
        F.element_at(
            F.col("wind_direction_10m_arr"),
            F.col("source_hour_index") + F.lit(1),
        ).cast("double")
    )
)


# ============================================================
# 7. CREATE SILVER GRAIN + DETERMINISTIC WEATHER KEY
# ============================================================

silver = (
    hourly
    .withColumn(
        "weather_date_local",
        F.to_date(
            F.col("weather_hour_local")
        )
    )
    .withColumn(
        "weather_hour_of_day_local",
        F.hour(
            F.col("weather_hour_local")
        ).cast("int")
    )
    .withColumn(
        "weather_key",
        F.sha2(
            F.concat_ws(
                "|",
                F.upper(
                    F.trim(
                        F.col("airport_code")
                    )
                ),
                F.date_format(
                    F.col("weather_hour_local"),
                    "yyyy-MM-dd HH:mm:ss",
                ),
            ),
            256,
        )
    )
    .select(
        "weather_key",
        "airport_code",

        "weather_date_local",
        "weather_hour_of_day_local",
        "weather_hour_local",
        "weather_hour_utc",

        "response_timezone",
        "response_timezone_abbreviation",
        "response_utc_offset_seconds",

        "source_latitude",
        "source_longitude",
        "source_elevation_m",

        "temperature_2m_c",
        "relative_humidity_2m_pct",
        "precipitation_mm",
        "snowfall_cm",
        "weather_code",
        "cloud_cover_pct",
        "wind_speed_10m_kmh",
        "wind_direction_10m_deg",

        "source_hour_index",

        "request_start_date",
        "request_end_date",

        *BRONZE_LINEAGE_COLS,

        F.lit(
            WEATHER_KEY_VERSION
        ).alias(
            "_weather_key_version"
        ),

        F.lit(
            SILVER_VERSION
        ).alias(
            "_silver_version"
        ),

        F.current_timestamp().alias(
            "_silver_transformed_at_utc"
        ),
    )
)


# ============================================================
# 8. PRE-WRITE SILVER QUALITY GATES
# ============================================================

silver_rows = silver.count()

print("\nSilver candidate rows:", silver_rows)

assert silver_rows == EXPECTED_TOTAL_ROWS, (
    f"STOP: expected {EXPECTED_TOTAL_ROWS} hourly rows "
    f"but found {silver_rows}"
)


# ------------------------------------------------------------
# Per-airport reconciliation
# ------------------------------------------------------------

airport_counts = (
    silver
    .groupBy("airport_code")
    .count()
    .orderBy("airport_code")
)

display(airport_counts)


airport_count_map = {
    r["airport_code"]: r["count"]
    for r in airport_counts.collect()
}

assert airport_count_map == {
    "ATL": EXPECTED_HOURS_PER_AIRPORT,
    "ORD": EXPECTED_HOURS_PER_AIRPORT,
}, (
    f"STOP: unexpected airport hourly counts: "
    f"{airport_count_map}"
)


# ------------------------------------------------------------
# Required grain fields cannot be NULL
# ------------------------------------------------------------

required_key_nulls = (
    silver
    .filter(
        F.col("weather_key").isNull()
        |
        F.col("airport_code").isNull()
        |
        F.col("weather_hour_local").isNull()
        |
        F.col("weather_hour_utc").isNull()
    )
    .count()
)

assert required_key_nulls == 0, (
    f"STOP: required weather-grain NULL rows = "
    f"{required_key_nulls}"
)


# ------------------------------------------------------------
# Local airport-hour uniqueness
# ------------------------------------------------------------

duplicate_local_groups = (
    silver
    .groupBy(
        "airport_code",
        "weather_hour_local",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_local_groups == 0, (
    f"STOP: duplicate local airport-hour groups = "
    f"{duplicate_local_groups}"
)


# ------------------------------------------------------------
# UTC airport-hour uniqueness
# ------------------------------------------------------------

duplicate_utc_groups = (
    silver
    .groupBy(
        "airport_code",
        "weather_hour_utc",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_utc_groups == 0, (
    f"STOP: duplicate UTC airport-hour groups = "
    f"{duplicate_utc_groups}"
)


# ------------------------------------------------------------
# Deterministic weather_key uniqueness
# ------------------------------------------------------------

distinct_weather_keys = (
    silver
    .select("weather_key")
    .distinct()
    .count()
)

assert distinct_weather_keys == silver_rows, (
    f"STOP: weather_key uniqueness failed: "
    f"{distinct_weather_keys} distinct keys "
    f"for {silver_rows} rows"
)


# ------------------------------------------------------------
# Date boundary validation
# ------------------------------------------------------------

outside_requested_period = (
    silver
    .filter(
        (F.col("weather_date_local") < F.col("request_start_date"))
        |
        (F.col("weather_date_local") > F.col("request_end_date"))
    )
    .count()
)

assert outside_requested_period == 0, (
    f"STOP: hourly rows outside requested period = "
    f"{outside_requested_period}"
)


# ------------------------------------------------------------
# Weather measurement NULL checks
# ------------------------------------------------------------

WEATHER_VALUE_COLS = [
    "temperature_2m_c",
    "relative_humidity_2m_pct",
    "precipitation_mm",
    "snowfall_cm",
    "weather_code",
    "cloud_cover_pct",
    "wind_speed_10m_kmh",
    "wind_direction_10m_deg",
]


weather_null_counts = (
    silver
    .agg(
        *[
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in WEATHER_VALUE_COLS
        ]
    )
    .first()
    .asDict()
)

weather_null_total = sum(
    weather_null_counts.values()
)

print(
    "\nWeather-value null counts:",
    weather_null_counts,
)

assert weather_null_total == 0, (
    f"STOP: weather-value NULL count = "
    f"{weather_null_total}"
)


# ------------------------------------------------------------
# Domain sanity checks
# ------------------------------------------------------------

range_failures = (
    silver
    .filter(
        (F.col("relative_humidity_2m_pct") < 0)
        |
        (F.col("relative_humidity_2m_pct") > 100)
        |
        (F.col("cloud_cover_pct") < 0)
        |
        (F.col("cloud_cover_pct") > 100)
        |
        (F.col("precipitation_mm") < 0)
        |
        (F.col("snowfall_cm") < 0)
        |
        (F.col("wind_speed_10m_kmh") < 0)
        |
        (F.col("wind_direction_10m_deg") < 0)
        |
        (F.col("wind_direction_10m_deg") > 360)
    )
    .count()
)

assert range_failures == 0, (
    f"STOP: weather domain-range failures = "
    f"{range_failures}"
)


# ------------------------------------------------------------
# Validate UTC conversion against source numeric offset
#
# April ORD/ATL contain no DST transition, so the
# Open-Meteo response offset should agree with IANA conversion.
# The IANA conversion remains the authoritative transformation.
# ------------------------------------------------------------

utc_check = (
    silver
    .withColumn(
        "_weather_hour_utc_from_offset",
        F.from_unixtime(
            F.unix_timestamp(
                F.col("weather_hour_local")
            )
            -
            F.col("response_utc_offset_seconds")
        ).cast("timestamp")
    )
)


utc_offset_mismatches = (
    utc_check
    .filter(
        F.col("weather_hour_utc")
        !=
        F.col("_weather_hour_utc_from_offset")
    )
    .count()
)

assert utc_offset_mismatches == 0, (
    f"STOP: IANA timezone vs source-offset UTC mismatches = "
    f"{utc_offset_mismatches}"
)


# ------------------------------------------------------------
# Bronze lineage remains present after explode
# ------------------------------------------------------------

silver_lineage_null_counts = (
    silver
    .agg(
        *[
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in BRONZE_LINEAGE_COLS
        ]
    )
    .first()
    .asDict()
)

assert sum(
    silver_lineage_null_counts.values()
) == 0, (
    "STOP: Silver weather lost required Bronze lineage"
)


print("\nPRE-WRITE SILVER WEATHER QA: PASS")


# ============================================================
# 9. WRITE SILVER WEATHER v0.1
#
# The target remains intentionally pilot-only.
# Do not append the other 13 airports yet.
# ============================================================

(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        TARGET_TABLE
    )
)


# ============================================================
# 10. POST-WRITE VALIDATION
# ============================================================

target = spark.table(TARGET_TABLE)

target_rows = target.count()

actual_types = dict(target.dtypes)

expected_types = {
    "weather_key": "string",
    "airport_code": "string",
    "weather_date_local": "date",
    "weather_hour_of_day_local": "int",
    "weather_hour_local": "timestamp",
    "weather_hour_utc": "timestamp",
    "temperature_2m_c": "double",
    "relative_humidity_2m_pct": "double",
    "precipitation_mm": "double",
    "snowfall_cm": "double",
    "weather_code": "int",
    "cloud_cover_pct": "double",
    "wind_speed_10m_kmh": "double",
    "wind_direction_10m_deg": "double",
}


type_failures = {
    col_name: (
        actual_types.get(col_name),
        expected_type,
    )
    for col_name, expected_type
    in expected_types.items()
    if actual_types.get(col_name) != expected_type
}


assert target_rows == EXPECTED_TOTAL_ROWS, (
    f"STOP: target has {target_rows} rows; "
    f"expected {EXPECTED_TOTAL_ROWS}"
)

assert not type_failures, (
    f"STOP: unexpected target data types: "
    f"{type_failures}"
)


post_duplicate_groups = (
    target
    .groupBy(
        "airport_code",
        "weather_hour_local",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert post_duplicate_groups == 0, (
    f"STOP: target duplicate airport-hour groups = "
    f"{post_duplicate_groups}"
)


# ============================================================
# 11. FINAL TASK 17 EVIDENCE
# ============================================================

print("\n" + "=" * 78)
print("AIROPS 360 - TASK 17 SILVER HOURLY WEATHER NORMALIZATION")
print("=" * 78)

print(f"Source table:                    {SOURCE_TABLE}")
print(f"Target table:                    {TARGET_TABLE}")
print(f"Silver version:                  {SILVER_VERSION}")

print()
print(f"Bronze response rows:            {bronze_rows}")
print(f"ORD expected hourly rows:        {EXPECTED_HOURS_PER_AIRPORT}")
print(f"ATL expected hourly rows:        {EXPECTED_HOURS_PER_AIRPORT}")
print(f"Silver hourly rows:              {target_rows}")

print()
print(f"Distinct weather_key values:     {distinct_weather_keys}")
print(f"Duplicate local airport-hours:   {duplicate_local_groups}")
print(f"Duplicate UTC airport-hours:     {duplicate_utc_groups}")
print(f"Required key NULL rows:          {required_key_nulls}")

print()
print(f"JSON parse failures:              {json_parse_failures}")
print(f"Rows outside requested period:   {outside_requested_period}")
print(f"Weather-value NULLs:             {weather_null_total}")
print(f"Weather domain failures:         {range_failures}")
print(f"UTC conversion mismatches:       {utc_offset_mismatches}")
print(f"Post-write duplicate groups:     {post_duplicate_groups}")
print(f"Type failures:                   {type_failures}")

print()
print(
    "Grain:                           "
    "1 airport + 1 local observation hour"
)

print("=" * 78)

print("\nTASK 17 STATUS: PASS")


# ============================================================
# 12. HUMAN-READABLE EVIDENCE
# ============================================================

print("\nHourly count by airport:")

display(
    target
    .groupBy(
        "airport_code",
        "response_timezone",
    )
    .agg(
        F.count("*").alias("hourly_rows"),
        F.min(
            "weather_hour_local"
        ).alias(
            "first_local_hour"
        ),
        F.max(
            "weather_hour_local"
        ).alias(
            "last_local_hour"
        ),
        F.min(
            "weather_hour_utc"
        ).alias(
            "first_utc_hour"
        ),
        F.max(
            "weather_hour_utc"
        ).alias(
            "last_utc_hour"
        ),
    )
    .orderBy(
        "airport_code"
    )
)


print("\nSilver hourly weather sample:")

display(
    target
    .select(
        "weather_key",
        "airport_code",
        "weather_hour_local",
        "weather_hour_utc",
        "response_timezone",
        "temperature_2m_c",
        "relative_humidity_2m_pct",
        "precipitation_mm",
        "snowfall_cm",
        "weather_code",
        "cloud_cover_pct",
        "wind_speed_10m_kmh",
        "wind_direction_10m_deg",
        "_bronze_batch_key",
        "_silver_version",
    )
    .orderBy(
        "airport_code",
        "weather_hour_local",
    )
    .limit(30)
)

StatementMeta(, c4621d14-bbd6-49eb-b090-55536f11e253, 3, Finished, Available, Finished, False)

TASK 17 CONFIGURATION
---------------------
Source table : lh_airops_bronze.brz_weather_api_raw
Target table : slv_weather_hourly
Airports     : ['ORD', 'ATL']
Period       : 2026-04-01 to 2026-04-30
Expected rows: 1440

Bronze pilot rows: 2
BRONZE PILOT VALIDATION: PASS

JSON PARSING: PASS


SynapseWidget(Synapse.DataFrame, 62f12c48-5156-41c9-807a-1299af732a0d)

TIMEZONE + ARRAY CARDINALITY VALIDATION: PASS

OPEN-METEO UNIT VALIDATION
unit_temperature_2m: ['°C']
unit_relative_humidity_2m: ['%']
unit_precipitation: ['mm']
unit_snowfall: ['cm']
unit_weather_code: ['wmo code']
unit_cloud_cover: ['%']
unit_wind_speed_10m: ['km/h']
unit_wind_direction_10m: ['°']
UNIT VALIDATION: PASS

Silver candidate rows: 1440


SynapseWidget(Synapse.DataFrame, 0bb239fc-2376-4bba-b166-177d73216368)


Weather-value null counts: {'temperature_2m_c': 0, 'relative_humidity_2m_pct': 0, 'precipitation_mm': 0, 'snowfall_cm': 0, 'weather_code': 0, 'cloud_cover_pct': 0, 'wind_speed_10m_kmh': 0, 'wind_direction_10m_deg': 0}

PRE-WRITE SILVER WEATHER QA: PASS

AIROPS 360 - TASK 17 SILVER HOURLY WEATHER NORMALIZATION
Source table:                    lh_airops_bronze.brz_weather_api_raw
Target table:                    slv_weather_hourly
Silver version:                  0.1

Bronze response rows:            2
ORD expected hourly rows:        720
ATL expected hourly rows:        720
Silver hourly rows:              1440

Distinct weather_key values:     1440
Duplicate local airport-hours:   0
Duplicate UTC airport-hours:     0
Required key NULL rows:          0

JSON parse failures:              0
Rows outside requested period:   0
Weather-value NULLs:             0
Weather domain failures:         0
UTC conversion mismatches:       0
Post-write duplicate groups:     0
Type failures:           

SynapseWidget(Synapse.DataFrame, 0ef55171-9df6-4be6-acce-bcd6fb87998e)


Silver hourly weather sample:


SynapseWidget(Synapse.DataFrame, 6790778f-b39e-40c0-ba56-f7f41e44d8f2)